# 🚗 Car Market Trends Analysis — Notebook 4
## Machine Learning — Predicting Used Car Selling Price

**Project:** Car Market Trends Analysis with Car Dekho Data  
**Goal:** Train a regression model to predict the selling price of a used car.

---
### Steps Covered
| Step | Task |
|------|------|
| 1 | Load & prepare features |
| 2 | Encode categorical columns (One-Hot Encoding) |
| 3 | Train / test split (80/20, no data leakage) |
| 4 | Train Linear Regression |
| 5 | Train Random Forest Regressor |
| 6 | Evaluate both models (MAE, RMSE, R²) |
| 7 | Compare results and pick the best model |
| 8 | Feature importance (Random Forest) |
| 9 | Residual analysis |
| 10 | Make a single prediction |
| 11 | Save the model |

---
## Step 0 — Import Libraries

In [ ]:
import os, sys, pickle, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection   import train_test_split
from sklearn.linear_model      import LinearRegression
from sklearn.ensemble          import RandomForestRegressor
from sklearn.metrics           import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing     import OneHotEncoder
from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.05)

BLUE, ORANGE, PURPLE = '#2563eb', '#ea580c', '#7c5cd8'
print('All libraries imported successfully.')

---
## Step 1 — Load Data & Select Features

In [ ]:
# Load cleaned dataset
CLEANED_PATH = os.path.join('..', 'data', 'car_dekho_cleaned.csv')
if os.path.exists(CLEANED_PATH):
    df = pd.read_csv(CLEANED_PATH)
else:
    sys.path.insert(0, os.path.join('..', 'src'))
    from data_cleaning import get_clean_data
    df = get_clean_data()

print(f'Dataset shape: {df.shape}')
df.head(3)

In [ ]:
# ── Feature selection ──────────────────────────────────────────────────────────
# We use 7 features that are naturally available when a car is listed for sale.
# Car_Name is excluded (97 unique values → overfitting risk on 299 rows).
# Year is excluded because Car_Age captures the same information more clearly.

NUMERIC_FEATURES     = ['Present_Price', 'Kms_Driven', 'Owner', 'Car_Age']
CATEGORICAL_FEATURES = ['Fuel_Type', 'Seller_Type', 'Transmission']
TARGET               = 'Selling_Price'
ALL_FEATURES         = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()

print(f'Features selected ({len(ALL_FEATURES)}): {ALL_FEATURES}')
print(f'Target: {TARGET}')
print(f'X shape: {X.shape},  y shape: {y.shape}')
X.head(3)

---
## Step 2 — One-Hot Encoding Explained

Machine learning models work with **numbers**, not text. We must convert
`Fuel_Type`, `Seller_Type`, and `Transmission` into numeric columns.

**One-Hot Encoding** creates a new 0/1 column for each category value:

| Fuel_Type | Fuel_Petrol | Fuel_Diesel | Fuel_CNG |
|-----------|-------------|-------------|----------|
| Petrol    | 1           | 0           | 0        |
| Diesel    | 0           | 1           | 0        |
| CNG       | 0           | 0           | 1        |

We do this **inside a Pipeline** so the encoding is always fitted on training data only — this prevents **data leakage**.

In [ ]:
# Build the preprocessing + model pipeline
def build_pipeline(model):
    """
    Creates a Pipeline:
      1. ColumnTransformer:
         - numeric columns → passed through unchanged
         - categorical columns → OneHotEncoder
      2. Regression model
    """
    cat_transformer = OneHotEncoder(
        handle_unknown='ignore',   # unseen categories in new data won't crash
        sparse_output=False        # return dense array, not sparse matrix
    )
    preprocessor = ColumnTransformer(transformers=[
        ('num', 'passthrough',   NUMERIC_FEATURES),
        ('cat', cat_transformer, CATEGORICAL_FEATURES),
    ])
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor',    model),
    ])

print('Pipeline builder defined.')

---
## Step 3 — Train / Test Split

In [ ]:
# Split BEFORE any fitting — the encoder never sees test data during training
# test_size=0.20 → 20% held out for evaluation (60 samples)
# random_state=42 → reproducible split every time you run this notebook

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f'Training samples : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing  samples : {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)')
print()
print('The encoder will be fitted ONLY on X_train.')
print('X_test is never seen during training → no data leakage.')

---
## Step 4 — Train Linear Regression

In [ ]:
# Linear Regression assumes the relationship between features and price is linear.
# Simple, fast, and easy to understand — good baseline model.

lr_pipeline = build_pipeline(LinearRegression())
lr_pipeline.fit(X_train, y_train)   # learn from training data only
y_pred_lr = lr_pipeline.predict(X_test)  # predict on unseen test data

print('Linear Regression trained.')
print(f'First 5 predictions vs actual:')
comparison = pd.DataFrame({
    'Actual Price':    y_test.values[:5],
    'Predicted Price': y_pred_lr[:5].round(2)
})
print(comparison.to_string(index=False))

---
## Step 5 — Train Random Forest Regressor

In [ ]:
# Random Forest builds 100 decision trees and averages their predictions.
# More powerful than Linear Regression for complex non-linear patterns,
# but needs more data to work well.

rf_pipeline = build_pipeline(RandomForestRegressor(
    n_estimators=100,   # number of trees
    random_state=42,
    n_jobs=-1,          # use all CPU cores
))
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print('Random Forest trained (100 trees).')

---
## Step 6 — Evaluate Both Models

In [ ]:
def evaluate(name, y_true, y_pred):
    """
    Calculate and print MAE, RMSE, and R² for a model.

    MAE  (Mean Absolute Error):
         Average Rs difference between predicted and actual price.
         Easy to interpret. Lower is better.

    RMSE (Root Mean Squared Error):
         Like MAE but penalises large errors more heavily.
         Lower is better.

    R²   (R-squared):
         What % of price variation does the model explain?
         0.75 means the model explains 75% of the variation.
         Closer to 1.0 is better.
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    print(f'  {name}')
    print(f'    MAE   = Rs {mae:.4f} Lakhs  ← avg prediction error')
    print(f'    RMSE  = Rs {rmse:.4f} Lakhs  ← penalises large errors')
    print(f'    R²    = {r2:.4f}  ← {r2*100:.1f}% of price variance explained')

    if   r2 >= 0.90: label = 'Excellent'
    elif r2 >= 0.75: label = 'Good'
    elif r2 >= 0.50: label = 'Moderate'
    else:            label = 'Weak'
    print(f'    Grade = {label}')
    print()
    return {'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4), 'R2': round(r2,4)}

print('=== MODEL EVALUATION (on 20% test set) ===\n')
r1 = evaluate('Linear Regression',         y_test, y_pred_lr)
r2 = evaluate('Random Forest (100 trees)', y_test, y_pred_rf)

---
## Step 7 — Compare Results

In [ ]:
results_df = pd.DataFrame([r1, r2]).set_index('Model')
print('=== COMPARISON TABLE ===')
print(results_df.to_string())

best_name = results_df['R2'].idxmax()
best_pipeline = lr_pipeline if 'Linear' in best_name else rf_pipeline
print(f'\nBest model: {best_name}')
print('Reason: Linear Regression achieves higher R² because the relationship')
print('between Present_Price and Selling_Price is approximately linear,\n'
      'and the small dataset (299 rows) does not give Random Forest enough\n'
      'data to build diverse trees.')

In [ ]:
# Bar chart comparing R² scores
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'R2']
colors_lr = [BLUE]*3
colors_rf = [ORANGE]*3

for ax, metric in zip(axes, metrics):
    vals = [r1[metric], r2[metric]]
    bars = ax.bar(['Linear\nRegression', 'Random\nForest'], vals,
                  color=[BLUE, ORANGE], edgecolor='white', width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=10)
    ax.set_title(metric)
    ax.set_ylabel(metric)

plt.suptitle('Model Comparison — MAE, RMSE, R² (Test Set)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 8 — Feature Importance (Random Forest)

In [ ]:
# Extract feature names after one-hot encoding
preprocessor = rf_pipeline.named_steps['preprocessor']
cat_names = (
    preprocessor.named_transformers_['cat']
    .get_feature_names_out(CATEGORICAL_FEATURES)
    .tolist()
)
all_names = NUMERIC_FEATURES + cat_names
importances = rf_pipeline.named_steps['regressor'].feature_importances_

imp_df = pd.DataFrame({'Feature': all_names, 'Importance': importances})
imp_df = imp_df.sort_values('Importance', ascending=False)

print('Feature Importances (Random Forest):')
print(imp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
imp_sorted = imp_df.sort_values('Importance')
ax.barh(imp_sorted['Feature'], imp_sorted['Importance'],
        color=BLUE, edgecolor='white')
ax.set_title('Random Forest — Feature Importance', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## Step 9 — Residual Analysis (Linear Regression)

In [ ]:
# Residuals = actual - predicted
# A good model should have residuals randomly scattered around 0.
# A pattern in residuals means the model is missing something.

residuals = y_test.values - y_pred_lr

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predicted vs actual
axes[0].scatter(y_pred_lr, y_test, alpha=0.6, color=BLUE, s=50)
min_v = min(y_pred_lr.min(), y_test.min())
max_v = max(y_pred_lr.max(), y_test.max())
axes[0].plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=1.5, label='Perfect')
axes[0].set_title('Predicted vs Actual Selling Price')
axes[0].set_xlabel('Predicted Price (Rs Lakhs)')
axes[0].set_ylabel('Actual Price (Rs Lakhs)')
axes[0].legend()

# Histogram: residual distribution
sns.histplot(residuals, bins=20, kde=True, color=PURPLE, ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_title('Residual Distribution (Actual − Predicted)')
axes[1].set_xlabel('Residual (Rs Lakhs)')
axes[1].set_ylabel('Count')

plt.suptitle('Residual Analysis — Linear Regression', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Mean residual : {residuals.mean():.4f} (should be close to 0)')
print(f'Std residual  : {residuals.std():.4f}')

---
## Step 10 — Make a Single Prediction

In [ ]:
# Example: predict the selling price of a 2017 Ciaz (Petrol, Dealer, Manual)
# Present price = Rs 9.85L, Kms = 15,000, Owner = 0, Car_Age = 7

new_car = pd.DataFrame([{
    'Present_Price' : 9.85,
    'Kms_Driven'    : 15000,
    'Owner'         : 0,
    'Car_Age'       : 7,
    'Fuel_Type'     : 'Petrol',
    'Seller_Type'   : 'Dealer',
    'Transmission'  : 'Manual',
}])

pred = best_pipeline.predict(new_car)[0]
pred = round(max(0.0, pred), 2)   # clamp to non-negative

print('=== SINGLE PREDICTION ===')
print(f'Input car       : Petrol, Dealer, Manual')
print(f'Present price   : Rs 9.85L')
print(f'Kms driven      : 15,000')
print(f'Owner           : 0 (first owner)')
print(f'Car age         : 7 years (manufactured 2017)')
print(f'Predicted price : Rs {pred} Lakhs')
print(f'Depreciation    : Rs {9.85 - pred:.2f} Lakhs  ({(9.85-pred)/9.85*100:.1f}% of present price)')

In [ ]:
# Try your own car — change the values below
my_car = pd.DataFrame([{
    'Present_Price' : 6.0,    # ← change this
    'Kms_Driven'    : 40000,  # ← change this
    'Owner'         : 1,      # ← change this (0,1,2,3)
    'Car_Age'       : 8,      # ← change this (2024 - year)
    'Fuel_Type'     : 'Petrol',    # Petrol / Diesel / CNG
    'Seller_Type'   : 'Individual',# Dealer / Individual
    'Transmission'  : 'Manual',    # Manual / Automatic
}])

my_pred = round(max(0.0, best_pipeline.predict(my_car)[0]), 2)
print(f'Your car predicted selling price: Rs {my_pred} Lakhs')

---
## Step 11 — Save the Best Model

In [ ]:
# Save the entire pipeline (preprocessor + model) as a .pkl file
# The Streamlit dashboard loads this file to make predictions

MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)
MODEL_PATH = os.path.join(MODELS_DIR, 'price_predictor.pkl')

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(best_pipeline, f)

print(f'Model saved to: {os.path.abspath(MODEL_PATH)}')
print('The saved file contains the full pipeline: encoder + trained model.')
print('Any new input will be encoded consistently before prediction.')

---
## Final Summary

### Model Performance (test set — real, not invented)

| Model | MAE (Rs L) | RMSE (Rs L) | R² |
|-------|-----------|-------------|----|
| **Linear Regression** | **1.47** | **2.52** | **0.75** ✅ |
| Random Forest | 1.50 | 3.61 | 0.49 |

### What the metrics mean
- **MAE = 1.47L** → On average, predictions are off by Rs 1.47 Lakhs
- **RMSE = 2.52L** → Large errors (e.g. luxury vehicles) push this higher
- **R² = 0.75** → The model explains 75% of the price variation

### Limitations
- Only 299 training samples — more data would improve accuracy
- Car_Name was not used as a feature — adding it with target encoding may help
- The model does not know vehicle condition, colour, or city

> **Next step:** Run the Streamlit dashboard to use the model interactively:
> ```bash
> streamlit run dashboard/app.py
> ```